# 🌍 NeuralShield v6 — Validation-Native: Native Alt Küme (MT vs Native)

**Amaç:** Eğitimdeki MT üstünlüğünün **çeviri artefaktı** olup olmadığını ölçmek. **Held-out native** veri (eğitimde YOK).

- Adaylar (held-out): tr (`beratcmn`, `hoatac`, `OnerAYTAS`), ru (`FaustSS`), ar/de (multilingual jailbreak setleri).
- Bu defter önce **şemayı basar** (kolon/split/label); sonra v6.2'yi native alt kümede ölçer.
- **Not:** Native benign bulunamazsa o dil AUC'siz (yalnız recall) raporlanır → sınırlama.

In [ ]:
# @title 1. Kurulum + Drive + Repo
import os, sys, json, re, gc, unicodedata
import warnings; warnings.filterwarnings('ignore')
os.environ['TOKENIZERS_PARALLELISM']='false'
try:
    from google.colab import drive; drive.mount('/content/drive')
except Exception as e: print('drive:', e)
!pip -q install -U transformers datasets scikit-learn langdetect 2>/dev/null
if not os.path.exists('ai-soc-agent-lab'):
    !git clone -q https://github.com/Mustafa-Caliskan/ai-soc-agent-lab.git
if os.path.exists('ai-soc-agent-lab'): os.chdir('ai-soc-agent-lab')
!git pull -q
import numpy as np, torch, pandas as pd
from datasets import load_dataset, get_dataset_split_names, get_dataset_config_names
MASTER='/content/drive/MyDrive/NeuralShield_v6_MasterData'
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

In [ ]:
# @title 2. Native held-out adayları yükle (şema-toleranslı)
from langdetect import detect, DetectorFactory
DetectorFactory.seed=42
SCRIPT={'ar':r'[\u0600-\u06FF]','ru':r'[\u0400-\u04FF]'}
def lang_of(t, expect=None):
    if expect: return expect
    for lg,pat in SCRIPT.items():
        if re.search(pat,t): return lg
    if re.search(r'[çğıöşüÇĞİÖŞÜ]',t): return 'tr'
    try:
        lg=detect(t[:400]); return lg if lg in ['tr','ru','ar','de'] else None
    except Exception: return None
TEXT_COLS=['text','prompt','instruction','question','query','inputs','input','jailbreak','content','sentence','prompt_ru','Prompt']
LABEL_COLS=['label','labels','type','class','is_attack','is_jailbreak','malicious','category','output']
POS={'1','1.0','attack','jailbreak','injection','prompt injection','malicious','unsafe','true','yes','harmful','injection_attack'}
CANDS=[('beratcmn/turkish-prompt-injections','tr'),('hoatac/prompt-injections-turkish','tr'),
       ('OnerAYTAS/Turkish_prompt_injection_jailbreak_dataset','tr'),
       ('FaustSS/russian_prompt_injections','ru'),
       ('wow2000/multilingual_jailbreak_challenges',None),
       ('darkknight25/Multilingual_Jailbreak_Dataset',None)]
ETH=re.compile(r'(suicide|intihar|genocide|soyk[ıi]r|ırkç|nefret söylemi|nuke|bomb|bomba|meth|heroin|ricin|sarin|nerve agent|uyuşturucu|fidye)',re.I)
def clean(t):
    t=str(t or '').strip(); return t if (15<=len(t)<=1500 and not ETH.search(t)) else None
rows=[]
for cid,expect in CANDS:
    try: splits=get_dataset_split_names(cid)
    except Exception as e: print('[i]',cid,'split yok:',str(e)[:60]); splits=['train']
    got=0; seen=set()
    for spl in splits[:2]:
        try: d=load_dataset(cid, split=spl)
        except Exception as e: print('   atla',spl,str(e)[:50]); continue
        df=d.to_pandas()
        if len(df)>60000: df=df.sample(60000,random_state=42)
        tcol=next((c for c in TEXT_COLS if c in df.columns), None)
        if not tcol:
            print('   [i]',cid,spl,'metin kolonu yok:',list(df.columns)[:6]); continue
        lcol=next((c for c in LABEL_COLS if c in df.columns), None)
        print(f'   OK {cid}/{spl} | n={len(df)} | tcol={tcol} lcol={lcol}')
        for _,r in df.iterrows():
            t=clean(r.get(tcol))
            if not t: continue
            lg=lang_of(t, expect)
            if lg not in ('tr','ru','ar','de'): continue
            lab=1 if (lcol is not None and str(r.get(lcol)).strip().lower() in POS) else 0
            k=(lg,re.sub(r'\s+',' ',unicodedata.normalize('NFKC',t).lower().strip()))
            if k in seen: continue
            seen.add(k); rows.append({'text':t,'label':lab,'language':lg,'source':cid}); got+=1
    print(f'  -> {cid}: {got} native ornek')
df=pd.DataFrame(rows)
if len(df):
    print('\n--- NATIVE DAGILIM (dil x label) ---'); print(df.groupby(['language','label']).size())
    json.dump(rows, open(MASTER+'/native_subset_v6.json','w',encoding='utf-8'), ensure_ascii=False, indent=1)
    print('-> native_subset_v6.json kaydedildi | N=',len(df))
else: print('Native veri bulunamadi (kaynaklar gated/şema farklı olabilir)')

In [ ]:
# @title 3. v6.2'yi native alt kümede değerlendir (+ Patronus)
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
from sklearn.metrics import roc_auc_score
def best_ckpt(base):
    if os.path.exists(base+'/config.json'): return base
    for d in [base+'/checkpoints', base]:
        if os.path.isdir(d):
            s=[x for x in os.listdir(d) if x.startswith('checkpoint-')]
            if s: return os.path.join(d, sorted(s,key=lambda x:int(x.split('-')[1]))[-1])
    return None
if len(df):
    texts=df['text'].tolist(); labs=df['label'].values
    def ev_ours(base):
        ck=best_ckpt(base)
        if not ck: return None
        tok=AutoTokenizer.from_pretrained(ck); mod=AutoModelForSequenceClassification.from_pretrained(ck,torch_dtype=torch.float16).to('cuda').eval(); pr=[]
        for i in range(0,len(texts),64):
            inp=tok(texts[i:i+64],return_tensors='pt',padding=True,truncation=True,max_length=512).to('cuda')
            with torch.no_grad(): pr.extend(torch.softmax(mod(**inp).logits.float(),-1)[:,1].cpu().numpy().tolist())
        del mod,tok; gc.collect(); torch.cuda.empty_cache(); return np.array(pr)
    P={'v6.2': ev_ours('/content/drive/MyDrive/NeuralShield_v6_mmbert/checkpoints_v6_2_backup')}
    try:
        clf=pipeline('text-classification',model='patronus-studio/wolf-defender-prompt-injection',truncation=True,max_length=2048,device=0,torch_dtype=torch.float16)
        res=clf(texts,batch_size=32); pos=('INJECTION','LABEL_1','ATTACK','UNSAFE','JAILBREAK','MALICIOUS')
        P['Patronus']=np.array([r['score'] if any(x in str(r['label']).upper() for x in pos) else 1-r['score'] for r in res])
        del clf; gc.collect(); torch.cuda.empty_cache()
    except Exception as e: print('Patronus atlandi:',str(e)[:70])
    out=[]
    for name,p in P.items():
        if p is None: continue
        for lg in ['tr','ru','ar','de']:
            g=np.where(df['language'].values==lg)[0]
            if len(g)==0: continue
            y=labs[g]; pr=p[g]; a=pr[y==1]; b=pr[y==0]
            rec=round(100*float((a>=0.7).mean()),1) if len(a) else None
            fp=round(100*float((b>=0.7).mean()),1) if len(b) else None
            auc=round(float(roc_auc_score(y,pr)),3) if (len(a) and len(b)) else None
            out.append({'Model':name,'dil':lg,'n':int(len(g)),'recall@0.7':rec,'fp@0.7':fp,'AUC':auc})
    res_df=pd.DataFrame(out); display(res_df)
    json.dump(out, open(MASTER+'/native_eval_v6.json','w',encoding='utf-8'), ensure_ascii=False, indent=2)
    print('-> native_eval_v6.json kaydedildi')
    print('\nNOT: AUC, o dilde hem attack hem benign varsa hesaplanir; yoksa None (yalniz recall).')
else: print('Native veri yok; once Cell 2 calistir.')